In [ ]:
# Dataset 인스턴스 생성
growth_stages = ["01"]
dataset = PlantDataset(root_dir=dataset_dir, stages=growth_stages, 
                       process_leaf=True, image_size=image_size,
                       side_view=side_view,
                       preload=False, image_processor=image_processor, add_sos_token=True)

In [ ]:
# Get encoder-decoder outputs for value prediction
batch = dataset[0]
labels = torch.tensor(batch['labels']).unsqueeze(0).to(model.device)
input_labels = labels[:,:-1]
expected = labels[:,1:]
pixel_values = batch['pixel_values'].unsqueeze(0).to(model.device)
with torch.no_grad():
    encoder_outputs = model.encoder(pixel_values)
    encoder_hidden_states = encoder_outputs.last_hidden_state
            # optionally project encoder_hidden_states
    if (
        model.encoder.config.hidden_size != model.decoder.config.hidden_size
        and model.decoder.config.cross_attention_hidden_size is None
    ):
        encoder_hidden_states = model.enc_to_dec_proj(encoder_hidden_states)
    decoder_outputs = model.decoder(
        input_ids=input_labels, # From <SOS>, without <EOS>
        encoder_hidden_states=encoder_hidden_states,
        output_hidden_states=True,
        output_attentions=True
    )

In [ ]:
estimated = torch.argmax(decoder_outputs.logits, dim=2)
# estimated.to("cpu").numpy()
print(input_labels)
input_labels_snp = input_labels[0][1:]
input_labels_snp[6] += 100

print(input_labels_snp)

#estimated = [input_labels_snp]

In [ ]:
# Check where is wrong
print(input_labels[0][1:])
print(estimated[0][:-1])
diff_mask = (input_labels[0][1:] == estimated[0][:-1])
print(diff_mask)

In [ ]:
print(decoder_outputs.cross_attentions[-1].shape)

last_cross_attention = decoder_outputs.cross_attentions[-1]
average_last_cross_attention = last_cross_attention.squeeze(0).mean(dim=0)

average_last_cross_attention = average_last_cross_attention[:,:-1]
num_patches = int(np.sqrt(average_last_cross_attention.size(-1)))
average_last_cross_attention_reshaped = average_last_cross_attention.reshape(-1, num_patches,num_patches)

In [ ]:
sum(last_cross_attention[0][0][0])

In [ ]:
print(average_last_cross_attention.shape)

In [ ]:
sum(average_last_cross_attention[0])

In [ ]:
print(average_last_cross_attention_reshaped.shape)
print(expected.shape)

In [ ]:
# Visulize along the word
import matplotlib.pyplot as plt
img = average_last_cross_attention.detach().cpu().numpy()
plt.imshow(img.transpose())

In [ ]:
# diff_mask에서 True인 인덱스 추출
true_indices = torch.where(diff_mask)[0].tolist()
print(true_indices)
false_indices = torch.where(diff_mask==False)[0].tolist()
print(false_indices)
# 시각화
for idx in false_indices:
    feature = average_last_cross_attention_reshaped[idx].cpu().numpy()
    backgroud = pixel_values.squeeze(0).permute(1, 2, 0).cpu().numpy()
    backgroud = cv2.normalize(np.array(backgroud), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    print(sum(sum(feature)))
    feature = cv2.resize(feature, (backgroud.shape[1], backgroud.shape[0]))
    # 이미지 표시
    plt.imshow(backgroud, alpha=0.5)  # 배경 이미지 먼저 표시
    plt.imshow(feature, alpha=0.5, cmap='viridis')  # 컬러맵 적용
    plt.colorbar()  # 컬러바 표시 (선택 사항)
    plt.title(f"Index: {idx}")  # 인덱스 표시
    plt.show()

In [ ]:
# diff_mask에서 True인 인덱스 추출
true_indices = torch.where(diff_mask)[0].tolist()
print(true_indices)
false_indices = torch.where(diff_mask==False)[0].tolist()
print(false_indices)
# 시각화
for idx in false_indices:
    feature = average_last_cross_attention_reshaped[idx].cpu().numpy()
    backgroud = pixel_values.squeeze(0).permute(1, 2, 0).cpu().numpy()
    backgroud = cv2.normalize(np.array(backgroud), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    feature = cv2.resize(feature, (backgroud.shape[1], backgroud.shape[0]))
    # print(sum(sum(feature)))
    # 이미지 표시
    plt.imshow(backgroud, alpha=0.5)  # 배경 이미지 먼저 표시
    plt.imshow(feature, alpha=0.5, cmap='viridis')  # 컬러맵 적용
    plt.colorbar()  # 컬러바 표시 (선택 사항)
    plt.title(f"Index: {idx}")  # 인덱스 표시
    plt.show()
    break


In [ ]:
import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from plant_tokenizer import SOS_TOKEN, EOS_TOKEN, PAD_TOKEN
from models.model import get_tgt_mask
from plant_dataset import PlantDataset, load_sideview_images
from image_process import process_leaf_image
from string_to_xml_to_vec import vec2xml, recursive_to_linked, pretty_print_xml
from plant_tokenizer import token2vec
import subprocess
import shutil

# Convert predictions to XML
temp_folder = "temp"
shutil.rmtree(temp_folder, ignore_errors=True)
os.makedirs(temp_folder, exist_ok=True)
plant_vec = token2vec(estimated[0].cpu().numpy()[5:])
plant_xml = vec2xml(plant_vec)
plant_xml_file_name = f"{temp_folder}/plant_{idx}_est.xml"
plant_xml = recursive_to_linked(plant_xml)
plant_xml_str = pretty_print_xml(plant_xml)
with open(plant_xml_file_name, "w") as f:
    f.write(plant_xml_str)

# Re-render predicted XML
re_render_xml(os.path.abspath(temp_folder), os.path.abspath(plant_xml_file_name))
if side_view:
    pred_img, _ = load_sideview_images(temp_folder, plant_xml_file_name.replace("xml","jpeg"), image_size, True)
else:
    pred_img = cv2.imread(plant_xml_file_name.replace("xml","jpeg"))
    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_RGB2BGR)
    leaf_area, plant_width, plant_height, leaf_img, _ = process_leaf_image(pred_img, sqaure_crop=True)
    pred_img = cv2.resize(leaf_img, (image_size, image_size))


plt.imshow(pred_img)

In [ ]:
import torch.nn.functional as F

original_outputs = model.encoder(pixel_values)
pred_img = cv2.cvtColor(pred_img, cv2.COLOR_RGB2BGR)
pred_pixel_vaules = image_processor(pred_img,return_tensors="pt").pixel_values[0].unsqueeze(0)
pred_pixel_vaules = pred_pixel_vaules.to(model.device)
rendered_outputs = model.encoder(pred_pixel_vaules)

# Get the full sequence of embeddings (all tokens, not just CLS)
original_embeddings = original_outputs.last_hidden_state  # Shape: [batch_size, sequence_length, hidden_size]
rendered_embeddings = rendered_outputs.last_hidden_state  # Shape: [batch_size, sequence_length, hidden_size]

# Normalize each token embedding
original_embeddings = F.normalize(original_embeddings, p=2, dim=2)  # Normalize each token embedding
rendered_embeddings = F.normalize(rendered_embeddings, p=2, dim=2)

# Compute attention/similarity matrix between all tokens from both images
# Shape: [batch_size, orig_seq_len, rend_seq_len]
token_similarities = torch.bmm(original_embeddings, rendered_embeddings.transpose(1, 2))

# Calculate the average diagonal similarity (for token-wise direct correspondence)
batch_size, orig_seq_len, rend_seq_len = token_similarities.shape
min_seq_len = min(orig_seq_len, rend_seq_len)

# Extract diagonal elements up to the minimum sequence length
diagonal_similarities = torch.zeros(batch_size, min_seq_len, device=model.device)
for b in range(batch_size):
    for i in range(min_seq_len):
        diagonal_similarities[b, i] = token_similarities[b, i, i]

max_similarities_per_token = token_similarities.max(dim=2)[0]  # Shape: [batch_size, orig_seq_len]
max_similarities_per_token = max_similarities_per_token[:,:-1].reshape(-1, num_patches, num_patches)
# Visualize
backgroud = pixel_values.squeeze(0).permute(1, 2, 0).cpu().numpy()
backgroud = cv2.normalize(np.array(backgroud), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)

diagonal_similarities_vis = diagonal_similarities[:,:-1].reshape(-1, num_patches, num_patches)
print(diagonal_similarities_vis.shape)
diagonal_similarities_vis = diagonal_similarities_vis.squeeze().detach().cpu().numpy()
diagonal_similarities_vis = cv2.resize(diagonal_similarities_vis, (pred_img.shape[1], pred_img.shape[0]))
print(diagonal_similarities.mean(dim=-1))

plt.imshow(backgroud, alpha=0.5)  # 배경 이미지 먼저 표시
#plt.imshow(pred_img, alpha=0.5)  # 배경 이미지 먼저 표시
plt.imshow(diagonal_similarities_vis, alpha=0.5, cmap='viridis')  # 컬러맵 적용
plt.colorbar()  # 컬러바 표시 (선택 사항)
plt.show()


plt.imshow(pred_img, alpha=0.5)  # 배경 이미지 먼저 표시
plt.imshow(diagonal_similarities_vis, alpha=0.5, cmap='viridis')  # 컬러맵 적용
plt.colorbar()  # 컬러바 표시 (선택 사항)
plt.show()

print(max_similarities_per_token.shape)
max_similarities_per_token = max_similarities_per_token.squeeze().detach().cpu().numpy()
max_similarities_per_token_vis = cv2.resize(max_similarities_per_token, (pred_img.shape[1], pred_img.shape[0]))
plt.imshow(backgroud, alpha=0.5)  # 배경 이미지 먼저 표시
#plt.imshow(pred_img, alpha=0.5)  # 배경 이미지 먼저 표시
plt.imshow(max_similarities_per_token_vis, alpha=0.5, cmap='viridis')  # 컬러맵 적용
plt.colorbar()  # 컬러바 표시 (선택 사항)
plt.show()

In [ ]:
# diagonal_similarities_vis = diagonal_similarities[:,:-1].reshape(-1, num_patches, num_patches)
# print(diagonal_similarities_vis.shape)
# feature = average_last_cross_attention_reshaped[false_indices[2]].cpu().numpy()
# print(feature.shape)
# diagonal_similarities_vis = diagonal_similarities_vis.squeeze().detach().cpu().numpy() * feature
# diagonal_similarities_vis = cv2.resize(diagonal_similarities_vis, (pred_img.shape[1], pred_img.shape[0]))

feature = average_last_cross_attention_reshaped[false_indices].mean(dim=0).cpu().numpy()
feature_vis = cv2.resize(feature, (pred_img.shape[1], pred_img.shape[0]))
print(feature.shape)
plt.imshow(backgroud, alpha=0.5)  # 배경 이미지 먼저 표시
#plt.imshow(pred_img, alpha=0.5)  # 배경 이미지 먼저 표시
plt.imshow(feature_vis, alpha=0.5, cmap='viridis')  # 컬러맵 적용
plt.colorbar()  # 컬러바 표시 (선택 사항)
plt.show()

In [ ]:
shoot_param_names = ["shoot_base_rotation_pitch", "shoot_base_rotation_yaw", "shoot_base_rotation_roll", "plant_age", "shoot_type"]
internode_param_names = ["internode_length", "internode_radius", "internode_pitch", "phyllotactic angle"]
petiole_param_names = ["petiole_length", "petiole_radius", "petiole_pitch", "petiole_curvature", "leaflet_scale"]
leaf_param_names = ["leaf_scale", "leaf pitch", "leaf yaw", "leaf roll"]
meta_param_names = ["vegetation fraction","width","height"]

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import imageio  # GIF 생성을 위한 라이브러리
from tqdm.auto import tqdm
from plant_tokenizer import SOS_TOKEN,EOS_TOKEN, META_TOKEN, PAD_TOKEN, token2vec

# GIF 파일 이름
gif_filename = "attention_heatmap.gif"

# GIF에 저장될 이미지 프레임 리스트
frames = []

# average_last_cross_attention_reshaped의 길이 (전체 프레임 수)
num_frames = len(average_last_cross_attention_reshaped)
prev_organ = None
for i in range(num_frames):
    # 1. 데이터 준비
    feature = average_last_cross_attention_reshaped[i].cpu().numpy() if hasattr(average_last_cross_attention_reshaped[i], 'cpu') else average_last_cross_attention_reshaped[i] # i번째 feature 추출, cpu() 호출 에러 방지
    backgroud = pixel_values.squeeze(0).permute(1, 2, 0).cpu().numpy() if hasattr(pixel_values, 'cpu') else pixel_values.squeeze(0).permute(1,2,0)
    backgroud = cv2.normalize(np.array(backgroud), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    # 2. feature 크기 조정
    feature = cv2.resize(feature, (backgroud.shape[1], backgroud.shape[0]))

    # 3. 히트맵 생성 (feature를 히트맵으로 변환)
    plt.imshow(backgroud, alpha=0.5)  # 배경 이미지 먼저 표시
    plt.imshow(feature, alpha=0.5, cmap='viridis')  # 컬러맵 적용
    plt.colorbar()  # 컬러바 표시 (선택 사항)
    # 4. 제목 설정
    title = f"[{i}]: "
    token = estimated[0][i].cpu().numpy()  # labels 리스트에서 제목 가져오기
    if token == SOS_TOKEN:
        title += "<SOS>"
    elif token == EOS_TOKEN:
        title += "<EOS>"
    elif token == PAD_TOKEN:
        title += "<PAD>"
    elif token == META_TOKEN:
        title += "<META>"
        prev_organ = "meta"
        param_count = 0
    else:
        if token < 24:
            depth = token // 6
            organ = token % 6
            title += f"Depth {depth} "
            param_count = 0
            if organ == 0:
                prev_organ = "shoot"
                title += "<SHOOT>"
            elif organ == 1:
                prev_organ = "internode"
                title += "<INTERNODE>"
            elif organ == 2:
                prev_organ = "petiole"
                title += "<PETIOLE>"
            else:
                prev_organ = "leaf"
                title += f"<LEAF{organ-3}>"
        else:
            if prev_organ:
                # title += f"{prev_organ} Param"
                if prev_organ == "shoot":
                    param_name = shoot_param_names[param_count]
                    value = token2vec([0,token])[0][2]
                elif prev_organ == "internode":
                    param_name = internode_param_names[param_count]
                    value = token2vec([1,token])[0][2]
                elif prev_organ == "petiole":
                    param_name = petiole_param_names[param_count]
                    value = token2vec([2,token])[0][2]
                elif prev_organ == "leaf":
                    param_name = leaf_param_names[param_count]
                    value = token2vec([3,token])[0][2]
                elif prev_organ == "meta":
                    param_name = meta_param_names[param_count]
                    value = token2vec([0,token])[0][2]
                else:
                    param_name = ""
                param_count += 1
            # print(token)
            title += f"{param_name}={value}"
    print(title)
    plt.title(title)  # 그래프 제목 설정
    fig = plt.gcf()
    fig.canvas.draw()
    heatmap_img = np.array(fig.canvas.renderer.buffer_rgba())
    plt.close(fig)
    heatmap_img = cv2.cvtColor(heatmap_img, cv2.COLOR_RGBA2BGR)
    heatmap_img = np.uint8(heatmap_img)


    # 6. 프레임 추가 (블렌딩된 이미지를 GIF 프레임으로 추가)
    # plt.imshow()로 표시하는 대신, 이미지를 저장하여 GIF 생성에 사용
    frames.append(heatmap_img)  # BGR 이미지를 직접 추가

# 7. GIF 생성
imageio.mimsave(gif_filename, frames, duration=0.5)  # duration: 프레임 간 간격 (초)

print(f"GIF 파일 '{gif_filename}' 생성 완료!")